<a href="https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsayeshaqamar/flyrank-mlinternship-ayesha/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import pandas as pd

# Load the March 2026 daily performance file
file_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df = pd.read_parquet(file_path)

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (9841378, 31)
Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [4]:
print("df is loaded:", "df" in globals())
print("Rows:", len(df))
print("Columns:", len(df.columns))

df is loaded: True
Rows: 9841378
Columns: 31


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Baseline rule

I will use a simple rule-based score to prioritize content pages for review.

The rule uses observed performance signals only:

1. **Low organic traffic** — pages with relatively low organic sessions receive a higher priority score.
2. **Low CTR** — pages with lower search click-through rate receive a higher priority score when enough impressions are available.

The purpose is not to predict a measured refresh outcome. The score is a **directional decision-support baseline** for deciding which pages deserve review first.

### Score

The baseline score has two components:

- Low organic traffic: 0–2 points
- Low CTR: 0–2 points

The total score ranges from **0 to 4**.

Higher scores mean higher review priority.

### Reason codes

- `LOW_ORGANIC_TRAFFIC` — the page has relatively low organic session volume.
- `LOW_CTR` — the page has relatively low search CTR.
- `LOW_TRAFFIC_AND_CTR` — both signals are present.
- `NO_FLAG` — neither signal crosses the rule threshold.

### Action labels

- `REVIEW_REFRESH` — higher-priority page for manual review.
- `MONITOR` — weaker signal; monitor before taking action.

This is a baseline heuristic, not a ground-truth refresh prediction.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 1
# Define the baseline rule and reason codes

LOW_TRAFFIC_POINTS = 2
LOW_CTR_POINTS = 2

print("Baseline rule:")
print("- Low organic traffic = 2 points")
print("- Low CTR = 2 points")
print("- Total score range = 0 to 4")
print()
print("Reason codes:")
print("- LOW_ORGANIC_TRAFFIC")
print("- LOW_CTR")
print("- LOW_TRAFFIC_AND_CTR")
print("- NO_FLAG")
print()
print("Action labels:")
print("- REVIEW_REFRESH")
print("- MONITOR")

Baseline rule:
- Low organic traffic = 2 points
- Low CTR = 2 points
- Total score range = 0 to 4

Reason codes:
- LOW_ORGANIC_TRAFFIC
- LOW_CTR
- LOW_TRAFFIC_AND_CTR
- NO_FLAG

Action labels:
- REVIEW_REFRESH
- MONITOR


## 2. Build the ranked queue

I aggregate the daily observations to the content-page level so that the queue contains one row per content page.

For each page, I calculate:

- total organic sessions
- total GSC impressions
- total GSC clicks
- CTR from clicks divided by impressions

The thresholds are calculated from the observed distribution rather than chosen from a future outcome.

A page receives:

- 2 points for low organic traffic when its organic sessions are at or below the 25th percentile.
- 2 points for low CTR when its CTR is at or below the 25th percentile among pages with at least 100 impressions.

The resulting score is used only for directional review prioritization.

The queue is ranked from highest score to lowest score and saved to `work/outputs/baseline_action_score.csv`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 2
# Build page-level features and the baseline ranked queue

import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Aggregate daily observations to content-page level
# ---------------------------------------------------------

page_df = (
    df.groupby("content_hash_id", as_index=False)
      .agg(
          organic_sessions=("sessions_organic", "sum"),
          gsc_impressions=("gsc_impressions", "sum"),
          gsc_clicks=("gsc_clicks", "sum"),
          observed_days=("report_date", "nunique")
      )
)

# ---------------------------------------------------------
# 2. Calculate CTR
# ---------------------------------------------------------

page_df["ctr"] = np.where(
    page_df["gsc_impressions"] > 0,
    page_df["gsc_clicks"] / page_df["gsc_impressions"],
    np.nan
)

# ---------------------------------------------------------
# 3. Calculate thresholds from observed data
# ---------------------------------------------------------

traffic_threshold = page_df["organic_sessions"].quantile(0.25)

ctr_threshold = page_df.loc[
    page_df["gsc_impressions"] >= 100,
    "ctr"
].quantile(0.25)

print("Organic-session threshold:", round(traffic_threshold, 4))
print("CTR threshold:", round(ctr_threshold, 4))

# ---------------------------------------------------------
# 4. Create rule flags
# ---------------------------------------------------------

page_df["low_organic_traffic"] = (
    page_df["organic_sessions"] <= traffic_threshold
)

page_df["low_ctr"] = (
    (page_df["gsc_impressions"] >= 100) &
    (page_df["ctr"] <= ctr_threshold)
)

# ---------------------------------------------------------
# 5. Calculate baseline score
# ---------------------------------------------------------

page_df["baseline_score"] = (
    page_df["low_organic_traffic"].astype(int) * LOW_TRAFFIC_POINTS
    +
    page_df["low_ctr"].astype(int) * LOW_CTR_POINTS
)

# ---------------------------------------------------------
# 6. Assign reason code
# ---------------------------------------------------------

def get_reason(row):
    if row["low_organic_traffic"] and row["low_ctr"]:
        return "LOW_TRAFFIC_AND_CTR"
    elif row["low_organic_traffic"]:
        return "LOW_ORGANIC_TRAFFIC"
    elif row["low_ctr"]:
        return "LOW_CTR"
    else:
        return "NO_FLAG"

page_df["reason_code"] = page_df.apply(get_reason, axis=1)

# ---------------------------------------------------------
# 7. Assign action label
# ---------------------------------------------------------

page_df["action"] = np.where(
    page_df["baseline_score"] >= 2,
    "REVIEW_REFRESH",
    "MONITOR"
)

# ---------------------------------------------------------
# 8. Rank the queue
# ---------------------------------------------------------

queue = page_df.sort_values(
    by=["baseline_score", "organic_sessions", "ctr"],
    ascending=[False, True, True]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# ---------------------------------------------------------
# 9. Select useful output columns
# ---------------------------------------------------------

queue_output = queue[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "action",
        "reason_code",
        "organic_sessions",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "observed_days"
    ]
]

# ---------------------------------------------------------
# 10. Write required CSV
# ---------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue_output.to_csv(output_path, index=False)

print("\nQueue created successfully.")
print("Number of pages:", len(queue_output))
print("CSV written to:", output_path)

print("\nAction counts:")
print(queue_output["action"].value_counts())

print("\nTop 10:")
display(queue_output.head(10))

Organic-session threshold: 0.0
CTR threshold: 0.0

Queue created successfully.
Number of pages: 331437
CSV written to: work/outputs/baseline_action_score.csv

Action counts:
action
REVIEW_REFRESH    289314
MONITOR            42123
Name: count, dtype: int64

Top 10:


,rank,content_hash_id,baseline_score,action,reason_code,organic_sessions,gsc_impressions,gsc_clicks,ctr,observed_days
0,1,content_0002bd310bf01f15,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,187,0,0.0,31
1,2,content_00040fc39626c7db,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,136,0,0.0,31
2,3,content_0004be2ef2278bd8,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,250,0,0.0,13
3,4,content_000f4b73532b9e9d,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,100,0,0.0,31
4,5,content_00121aaf9fd41914,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,315,0,0.0,24
5,6,content_00157b715bf9db77,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,3335,0,0.0,31
6,7,content_001689f50648eef1,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,326,0,0.0,31
7,8,content_0017cf2a6ec5895a,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,208,0,0.0,31
8,9,content_001802b49adbe9ea,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,1010,0,0.0,31
9,10,content_001981bb15c26040,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,0.0,317,0,0.0,31


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

The top 20 rows are the highest-ranked pages under this baseline rule.

These are not confirmed refresh candidates. They are pages selected for manual review because their observed signals match the rule.

For each page, I record:

- the action suggested by the rule,
- the reason code,
- a confidence note based on the available evidence,
- and what could make the recommendation wrong.

The "what would make it wrong" note is important because low traffic or low CTR alone does not prove that a page needs refreshing.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 3
# Generate the top-20 review table

top20 = queue_output.head(20).copy()

def confidence_note(row):
    if row["gsc_impressions"] >= 1000 and row["observed_days"] >= 20:
        return "More evidence available from impressions and observed days."
    elif row["gsc_impressions"] >= 100:
        return "Moderate evidence; review before action."
    else:
        return "Limited GSC evidence; treat cautiously."

def wrong_if(row):
    if row["reason_code"] == "LOW_ORGANIC_TRAFFIC":
        return "It may be low-volume because the page has a naturally narrow audience."
    elif row["reason_code"] == "LOW_CTR":
        return "Low CTR may reflect query intent, SERP position, or limited impressions."
    elif row["reason_code"] == "LOW_TRAFFIC_AND_CTR":
        return "Both signals may be low because the page has limited search demand."
    else:
        return "The rule may miss other reasons a page needs review."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

display(
    top20[
        [
            "rank",
            "content_hash_id",
            "action",
            "reason_code",
            "baseline_score",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,rank,content_hash_id,action,reason_code,baseline_score,confidence_note,what_would_make_it_wrong
0,1,content_0002bd310bf01f15,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
1,2,content_00040fc39626c7db,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
2,3,content_0004be2ef2278bd8,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
3,4,content_000f4b73532b9e9d,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
4,5,content_00121aaf9fd41914,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
5,6,content_00157b715bf9db77,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,More evidence available from impressions and o...,Both signals may be low because the page has l...
6,7,content_001689f50648eef1,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
7,8,content_0017cf2a6ec5895a,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...
8,9,content_001802b49adbe9ea,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,More evidence available from impressions and o...,Both signals may be low because the page has l...
9,10,content_001981bb15c26040,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,4,Moderate evidence; review before action.,Both signals may be low because the page has l...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

The baseline is intentionally simple, so some high-ranked pages may be weak picks.

A low traffic or low CTR signal does not prove that a page should be refreshed. For example, a page may naturally have low search demand, may target a narrow topic, or may have insufficient GSC evidence.

I therefore treat the queue as a manual-review list rather than a confirmed refresh list.

### Leakage check

The baseline uses only observed performance fields:

- `sessions_organic`
- `gsc_impressions`
- `gsc_clicks`
- `report_date` only through aggregation/observed-day counting

It does not use:

- a refresh label,
- a future performance outcome,
- a post-refresh result,
- client names,
- or product-specific flags.

Therefore, the score is a rule-based baseline rather than a label-derived model.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 Section 4
# Weak-pick inspection and leakage check

print("WEAK PICK CHECK")
print("================")

# Show the lowest-evidence rows among the top 20
weak_picks = top20.sort_values(
    by=["gsc_impressions", "observed_days"],
    ascending=[True, True]
).head(5)

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_score",
            "action",
            "reason_code",
            "gsc_impressions",
            "observed_days"
        ]
    ]
)

print("\nLEAKAGE CHECK")
print("=============")

used_fields = {
    "sessions_organic",
    "gsc_impressions",
    "gsc_clicks",
    "report_date",
    "content_hash_id"
}

forbidden_terms = [
    "refresh_needed",
    "refresh_priority",
    "refresh_outcome",
    "post_refresh",
    "future",
    "label"
]

print("Fields used by the baseline:")
for field in sorted(used_fields):
    print("-", field)

print("\nDirect refresh label present in source columns:",
      any("refresh" in col.lower() for col in df.columns))

print("Potential label-derived terms in used fields:",
      [term for term in forbidden_terms if any(term in field.lower()
                                               for field in used_fields)])

print("\nBaseline uses observed performance signals only.")

WEAK PICK CHECK


,rank,content_hash_id,baseline_score,action,reason_code,gsc_impressions,observed_days
3,4,content_000f4b73532b9e9d,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,100,31
15,16,content_002ab606fe74b46f,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,105,31
13,14,content_00280d32b6d5df0b,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,114,31
1,2,content_00040fc39626c7db,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,136,31
14,15,content_00290defada3742a,4,REVIEW_REFRESH,LOW_TRAFFIC_AND_CTR,150,31



LEAKAGE CHECK
Fields used by the baseline:
- content_hash_id
- gsc_clicks
- gsc_impressions
- report_date
- sessions_organic

Direct refresh label present in source columns: False
Potential label-derived terms in used fields: []

Baseline uses observed performance signals only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.